# 冻结小模型的动态风险层对照

不重新训练，保留源目标和开盘路由，只用原合法约束所允许的最大beta旁路动态风险上限；回撤门仍为30%。诊断不是上线建议，不扫阈值。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.backtest.risk_cap_ablation import LegalCapOpenPolicy
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.minute_intake import atomic_json
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/tangent_risk_layer_20261014';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';qlib.init(provider_uri=str(provider),region='us',kernels=1)
frozen,fa=_source(root,plan['frozen_source'],['inputs/plan','analysis/targets',*[f'analysis/native/{family}_{cost}' for family in ['candidate','soxx'] for cost in plan['slippage_bps']]])
source,sa=_source(root,plan['source'],['inputs/frame']);controller,ca=_source(root,plan['price_source'],['inputs/controller','inputs/plan']);action,aa=_source(root,plan['action_source'],['inputs/plan','controller/source_recovery_selection'])
for key in ['tariff','initial_cash','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:
 assert frozen['inputs/plan'][key]==plan[key]
frame=source['inputs/frame'];frame=frame.loc[frame.signal_date.between(plan['start'],plan['end'])].copy();targets=frozen['analysis/targets']['candidate'].reindex(pd.DatetimeIndex(frame.signal_date));assert targets.notna().all().all()
for col in targets:frame[col]=targets[col].to_numpy()
spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);legal_cap=1-plan['minimum_cash_weight']+2*plan['maximum_soxl_weight'];assert action['controller/source_recovery_selection']['action_beta']<=legal_cap
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,recorder_name='fixed_dynamic_cap_bypass',uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'frozen':fa,'prices':sa,'controller':ca,'action':aa},frame=frame);rec.save_objects(artifact_path='sources',policy=(root/'src/backtest/risk_cap_ablation.py').read_text());rec.log_params(new_fits=0,maximum_new_native_cases=3,legal_beta_cap=legal_cap)
books={};audits={}
for cost in plan['slippage_bps']:
 policy=LegalCapOpenPolicy(open_model=controller['inputs/controller']['frozen_open_action_value_model'],action_spec=spec,maximum_soxl_weight=plan['maximum_soxl_weight'],minimum_cash_weight=plan['minimum_cash_weight'])
 replay=replay_full_path(frame,models=[policy],plan=plan,slippage_bps=cost)
 assert not replay['replay'].recovery_action.any()
 book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=plan['tariff'],slippage_bps=cost,initial_cash=plan['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets'];books[('bypass',cost)]=book;audits[str(cost)]=pd.DataFrame(policy.audit)
 for family,key in [('full_stack','candidate'),('soxx','soxx')]:books[(family,cost)]=frozen[f'analysis/native/{key}_{cost}']
 rec.save_objects(artifact_path='analysis/native',**{f'{f}_{cost}':books[(f,cost)] for f in ['bypass','full_stack','soxx']});rec.log_metrics(completed_new_native_cases=len(audits));print('account',cost,flush=True)
rows=[];annual=[];intervals=[]
for (family,cost),book in books.items():
 path=book['authority'];bench=books[('soxx',cost)]['authority'];assert path.trade_date.equals(bench.trade_date);relative=np.log1p(path.strategy_return)-np.log1p(bench.strategy_return)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,bench),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'average_cash_pct':100*path.ending_cash_weight.mean(),'average_effective_beta':float((3*path.ending_soxl_weight+path.ending_soxx_weight).mean()),'orders':len(book['orders']),'broker_fees_usd':path.broker_and_external_fees_usd.sum(),'slippage_usd':path.slippage_reserve_usd.sum(),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean()})
 annual.append(annual_paired_metrics(path,bench).assign(family=family,slippage_bps=cost))
 if family=='bypass':
  for comparator in ['soxx','full_stack']:intervals.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261014)})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);intervals=pd.DataFrame(intervals);candidate=table.loc[table.family.eq('bypass')]
gates={'net_soxx_excess_all_costs':bool(candidate.relative_wealth.gt(1+1e-8).all()),'drawdown_30pct':bool(candidate.max_drawdown_pct.ge(-30).all() and candidate.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(candidate.cvar5_pct.ge(candidate.benchmark_cvar5_pct).all()),'rolling_70pct':bool(candidate.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual.loc[annual.family.eq('bypass')].groupby('slippage_bps').relative_wealth.apply(lambda x:x.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(intervals.lower_bound_positive.all()),'independent_development':False,'qualified_forward':False}
summary={'experiment':card.name,'decision':'fixed_risk_layer_diagnostic_no_promotion','gates':gates,'objective_achieved':False,'new_fits':0,'new_native_cases':len(audits),'reused_native_cases':6,'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'scope':plan['scope'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(tracking),'recorder_local_dir':str(tracking/str(rec.experiment_id)/rec.id),'qlib_docs':'https://qlib.readthedocs.io/en/latest/component/strategy.html','artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','sources/*','analysis/*']}}
rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=intervals,open_decisions=audits,summary=summary);artifacts=Path(summary['recorder_local_dir'])/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=summary['experiment_id']);atomic_json(summary,card/'summary.json');table.to_csv(card/'backtest_benchmark_comparison.csv',index=False);annual.to_csv(card/'annual_metrics.csv',index=False);intervals.to_csv(card/'uncertainty.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'recorder_uri':str(tracking),'artifact_policy':summary['artifact_policy']},sort_keys=False));assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];print(table[['family','slippage_bps','return_pct','max_drawdown_pct','monitored_drawdown_pct','average_cash_pct']].to_string(index=False));print(gates)


# 冻结小模型的风险层对照：收益提高但约50%回撤，不可晋升

本轮0新增拟合，复用已训练的小配置头及其固定三seed平均目标，新增0/2/5bp三套完整Qlib原生账户。保留来源EMA/accepted目标序列、原开盘执行模型、SOXL≤30%及CASH≥15%的合法仓位约束、10万美元起点及原IBKR Tiered费用投影。

诊断只用合法仓位约束推出的最大名义beta：1−0.15+2×0.30=1.45，替代动态风险上限及风险状态（设为0）。开盘执行路由仍使用当前账户持仓、收益历史和已知净值；违法持仓强制修复，合法小额HOLD可以不下单。恢复动作的目标beta上限0.6不可能超过1.45，故此条件下不触发恢复。没有扫描风险阈值，也不修改30%评价门。

这是风险上限/状态及其触发恢复机制的组合干预。原开盘分类器接收的cap/state也随干预改变，可能超出其训练分布；差额不能解释为某一个阈值的独立贡献。当前策略生产配置未改，此旁路不是上线建议。

|策略|滑点|累计净收益|收盘最大回撤|盘中监控最大回撤|平均现金|名义有效beta|
|---|---|---|---|---|---|---|
|动态风险层旁路|0bp|+436.91%|-49.28%|-49.82%|33.39%|1.170|
|完整小模型策略|0bp|+342.52%|-24.69%|-27.59%|43.27%|0.910|
|SOXX|0bp|+387.75%|-42.87%|-43.38%|0.15%|0.999|
|动态风险层旁路|2bp|+435.22%|-49.31%|-49.86%|33.39%|1.170|
|完整小模型策略|2bp|+335.36%|-24.74%|-27.64%|43.67%|0.899|
|SOXX|2bp|+387.73%|-42.87%|-43.39%|0.14%|0.999|
|动态风险层旁路|5bp|+432.92%|-49.39%|-49.94%|33.39%|1.170|
|完整小模型策略|5bp|+348.06%|-24.83%|-27.73%|43.67%|0.899|
|SOXX|5bp|+387.70%|-42.88%|-43.39%|0.12%|0.999|

比较覆盖944个执行区间，2023-01-03信号起至2026-10-09结算，所有成本情景各自持仓、计费和执行。2bp旁路较完整策略提高约99.86个累计收益百分点，却增加约22.22个监控回撤百分点。5bp同样约49.94%监控回撤，不能将它忽略或通过修改30%门槛宣称优化成功。

名义beta按3×SOXL权重+SOXX权重计算，不是估计得到的统计beta。旁路的平均现金约33.39%、名义beta约1.17，高于完整策略约0.90；收益超过SOXX可能含更高风险暴露的补偿，未做同暴露反事实，因此不能宣称模型有独立预测alpha。SOXX自身监控回撤约43.39%，旁路风险也高于SOXX。

|信号年度|旁路2bp净收益|相对SOXX财富变化|旁路5bp净收益|相对SOXX财富变化|
|---|---|---|---|---|
|2023|+60.56%|+2.52%|+60.44%|+2.47%|
|2024|+20.17%|-0.67%|+20.04%|-0.78%|
|2025|+53.79%|+4.76%|+53.53%|+4.58%|
|2026|+80.39%|+2.86%|+80.23%|+2.76%|

相对财富为增长倍数之比减1，不是收益百分点。无动态风险层账户的0/2/5bp收益较接近且逐步下降，但这只是一组历史情景，不能外推所有未来成本或风险状态稳定。完整策略的成本路径也不是简单从同一收益序列扣费用。

决定：fixed_risk_layer_diagnostic_no_promotion。净SOXX累计收益门通过不抵消约50%回撤；固定风险约束和CVaR门失败，滚动稳定性与区块置信也未通过（年度正超额比例门通过）；不晋升、不替换现役、不继续搜索风险上限/恢复阈值挽救。当前小模型的收益与回撤缺口需要同时解决，不能把放大风险暴露当作达到目标。历史已复用，没有独立开发/前向资格。

下一项信息需求是区分模型择时价值与平均风险暴露补偿，并找到能在相近风险下保留收益的证据；不先假定更复杂风控或继续砍现金就能做到。现有源目标和全部账户继续保留，稳定净超额目标未完成。

验证：开盘策略未来字段不变性、合法HOLD和违规持仓强制修复合同1 passed；三个新账户经原生Qlib现金、数量、逐笔费用和净值对账，和对照采用同日历。NAS冷读回见readback_verification.json。

[完整比较](backtest_benchmark_comparison.csv) · [年度](annual_metrics.csv) · [区块区间](uncertainty.csv) · [固定计划](plan.json)

Recorder `169645f3487a419ebe477bfae42f02b2`。
